# Effect of replacing pre-layer-2 activations with a spline approximation

This notebook trains the same phase-based spline model used in Notebook 22, but fits it directly in the full residual-stream activation space. It then replaces the activation before layer 2 with the spline prediction during evaluation.

The clean and intervened models are evaluated on exactly the same held-out examples. The reported metrics are:

- **Overall accuracy:** accuracy over all next-token predictions.
- **Newline precision:** accuracy when the model predicts `_NEWLINE_`.
- **Newline recall:** accuracy when the true target is `_NEWLINE_`.
- **Newline prediction frequency:** fraction of positions predicted as `_NEWLINE_`.

The intervention is applied at every position before layer 2. This is a full-space replacement, not a replacement using only the 3D PCA reconstruction.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.preprocessing import SplineTransformer
from IPython.display import display

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
N_TRAIN_EXAMPLES = 2_000
N_EVAL_EXAMPLES = 1_000
COLLECTION_BATCH_SIZE = 128
TRAIN_SEED = 23_001
EVAL_SEED = 23_002
N_SPLINE_KNOTS = 12
RIDGE_ALPHA = 1.0


In [ ]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()

n_layers = cfg["model"]["n_layers"]
if n_layers < 3:
    raise ValueError("This experiment requires a layer 2 attention block.")
PATCH_LAYER = 2
PATCH_SITE = f"blocks.{PATCH_LAYER}.hook_resid_pre"
seq_len = cfg["task"]["seq_len"]
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Intervention site: {PATCH_SITE}")


## Collect training activations and held-out evaluation examples

In [ ]:
def character_counts_for_tokens(token_rows):
    all_counts = []
    for row in token_rows.detach().cpu().tolist():
        running_count = 0
        row_counts = []
        for token_id in row:
            if token_id == vocab.newline_id:
                running_count = 0
            else:
                running_count += project_data.extract_character_count(vocab.decode_token(token_id))
            row_counts.append(running_count)
        all_counts.append(row_counts)
    return torch.tensor(all_counts, dtype=torch.float32)

@torch.no_grad()
def collect_training_activations(n_examples, seed):
    activation_chunks, count_chunks, width_chunks = [], [], []
    collected = 0
    batch_number = 0
    while collected < n_examples:
        size = min(COLLECTION_BATCH_SIZE, n_examples - collected)
        batch = project_data.make_batch(
            batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device,
            seed=seed + batch_number * COLLECTION_BATCH_SIZE,
        )
        _, cache = model.run_with_cache(
            batch.tokens, prepend_bos=False,
            names_filter=lambda name: name == PATCH_SITE,
        )
        activation_chunks.append(cache[PATCH_SITE].detach().cpu())
        count_chunks.append(character_counts_for_tokens(batch.tokens))
        width_chunks.append(batch.line_lengths.detach().cpu().float())
        collected += size
        batch_number += 1
    return (
        torch.cat(activation_chunks, dim=0)[:n_examples].numpy(),
        torch.cat(count_chunks, dim=0)[:n_examples].numpy(),
        torch.cat(width_chunks, dim=0)[:n_examples].numpy(),
    )

X_train, counts_train, widths_train = collect_training_activations(N_TRAIN_EXAMPLES, TRAIN_SEED)
print("Training activation tensor:", X_train.shape)


## Fit the full-space spline approximation

In [ ]:
width_mean = widths_train.mean()
width_std = max(widths_train.std(), 1.0)
phase_spline = SplineTransformer(
    n_knots=N_SPLINE_KNOTS, degree=3, include_bias=False,
)
phase_spline.fit(np.linspace(0.0, 1.5, 500)[:, None])

spline_feature_count = phase_spline.n_features_out_

def make_features(character_counts, line_lengths):
    n_examples, n_positions = character_counts.shape
    phase = np.clip(character_counts / line_lengths[:, None], 0.0, 1.5)
    line_length_centered = (line_lengths - width_mean) / width_std
    boundary_residual = (character_counts - line_lengths[:, None]) / max(width_mean, 1.0)
    token_position = np.broadcast_to(
        np.arange(n_positions, dtype=np.float32) / max(n_positions - 1, 1),
        (n_examples, n_positions),
    )
    spline_features = phase_spline.transform(phase.reshape(-1, 1))
    line_feature = np.repeat(line_length_centered, n_positions)[:, None]
    return np.concatenate([
        spline_features,
        spline_features * line_feature,
        boundary_residual.reshape(-1, 1),
        token_position.reshape(-1, 1),
    ], axis=1)

F_train = make_features(counts_train, widths_train)
trajectory_model = Ridge(alpha=RIDGE_ALPHA)
trajectory_model.fit(F_train, X_train.reshape(-1, X_train.shape[-1]))
print("Feature matrix:", F_train.shape)
print("Full-space activation dimension:", X_train.shape[-1])
print("Training reconstruction R²:", trajectory_model.score(F_train, X_train.reshape(-1, X_train.shape[-1])))


## Evaluate clean and spline-replaced models

In [ ]:
@torch.no_grad()
def evaluate_model(replace_pre_layer2=False):
    correct = 0
    total = 0
    true_positives = 0
    actual_newlines = 0
    predicted_newlines = 0
    batches_used = 0

    for start in range(0, N_EVAL_EXAMPLES, COLLECTION_BATCH_SIZE):
        size = min(COLLECTION_BATCH_SIZE, N_EVAL_EXAMPLES - start)
        batch = project_data.make_batch(
            batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device,
            seed=EVAL_SEED + start,
        )
        fwd_hooks = []
        if replace_pre_layer2:
            counts = character_counts_for_tokens(batch.tokens).numpy()
            widths = batch.line_lengths.detach().cpu().numpy().astype(np.float32)
            features = make_features(counts, widths)
            replacement = trajectory_model.predict(features).reshape(size, seq_len, -1)
            replacement = torch.as_tensor(replacement, device=device, dtype=model.W_Q.dtype)

            def replace_activation(activation, hook, replacement=replacement):
                return replacement.to(device=activation.device, dtype=activation.dtype)

            fwd_hooks = [(PATCH_SITE, replace_activation)]

        logits = model.run_with_hooks(
            batch.tokens, prepend_bos=False, fwd_hooks=fwd_hooks,
        )
        predictions = logits.argmax(dim=-1)
        targets = batch.targets
        target_newline = targets == vocab.newline_id
        prediction_newline = predictions == vocab.newline_id
        correct += int((predictions == targets).sum().item())
        total += predictions.numel()
        true_positives += int((target_newline & prediction_newline).sum().item())
        actual_newlines += int(target_newline.sum().item())
        predicted_newlines += int(prediction_newline.sum().item())
        batches_used += 1

    return {
        "overall_accuracy": correct / total,
        "newline_precision": true_positives / predicted_newlines if predicted_newlines else np.nan,
        "newline_recall": true_positives / actual_newlines if actual_newlines else np.nan,
        "newline_prediction_frequency": predicted_newlines / total,
        "correct": correct,
        "total": total,
        "true_positive_newlines": true_positives,
        "actual_newlines": actual_newlines,
        "predicted_newlines": predicted_newlines,
        "batches": batches_used,
    }

clean_metrics = evaluate_model(False)
patched_metrics = evaluate_model(True)
results = pd.DataFrame([
    {"condition": "clean", **clean_metrics},
    {"condition": "full-space spline replacement", **patched_metrics},
])
display(results[[
    "condition", "overall_accuracy", "newline_precision", "newline_recall",
    "newline_prediction_frequency", "actual_newlines", "predicted_newlines",
]].style.format({
    "overall_accuracy": "{:.4f}",
    "newline_precision": "{:.4f}",
    "newline_recall": "{:.4f}",
    "newline_prediction_frequency": "{:.4f}",
}))


## Change relative to the clean model

In [ ]:
metric_names = [
    "overall_accuracy",
    "newline_precision",
    "newline_recall",
    "newline_prediction_frequency",
]
change_table = pd.DataFrame({
    "metric": metric_names,
    "clean": [clean_metrics[name] for name in metric_names],
    "spline_replacement": [patched_metrics[name] for name in metric_names],
})
change_table["absolute_change"] = (
    change_table["spline_replacement"] - change_table["clean"]
)
change_table["relative_change"] = (
    change_table["absolute_change"] / change_table["clean"].replace(0, np.nan)
)
display(change_table.style.format({
    "clean": "{:.4f}",
    "spline_replacement": "{:.4f}",
    "absolute_change": "{:+.4f}",
    "relative_change": "{:+.2%}",
}))
